# Semantic Chunking

Groups one PDF's unified document (dual-pipeline merged text + Stage 1
figure captions, from `image_understanding.ipynb`) into topically
coherent chunks, using embedding similarity rather than heading
structure or LLM-judged continuity.

Each `UnifiedItem` (a page's transcribed text, or a `[FIGURE:...]`
caption block) is embedded as one unit, in document order. A chunk
boundary is placed wherever the cosine distance between consecutive
item embeddings exceeds a percentile-based threshold over the whole
document — sections that drift topic sharply split apart; a figure
immediately following the text it illustrates stays in the same chunk.

See `src/ingestion/semantic_chunk.py::SemanticChunker`. Output chunks
are the unit fed to concept/relation extraction downstream.

## Setup

In [ ]:
import json
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent  # notebook lives in notebooks/
sys.path.insert(0, str(PROJECT_ROOT / "src"))

from ingestion.semantic_chunk import SemanticChunker
from ingestion.unify import build_unified_items_from_merged, render_unified_markdown

PROJECT_ROOT

## Rebuild the unified document

`image_understanding.ipynb` writes the rendered text/markdown but not
the underlying `UnifiedItem` list, so it's rebuilt here from the two
artifacts that stage already produced: the dual-pipeline merged
document and the Stage 1 caption results.

In [ ]:
PDF_STEM = "Anatomy of Neck - Basic of DEMN.pdf_origin"
OUTPUT_DIR = PROJECT_ROOT / "output" / PDF_STEM / "auto"
MERGED_PATH = OUTPUT_DIR / f"{PDF_STEM}_dual_pipeline_merged.json"
CAPTIONS_PATH = OUTPUT_DIR / f"{PDF_STEM}_stage1_captions.json"

with open(MERGED_PATH) as f:
    merged_document = json.load(f)

with open(CAPTIONS_PATH) as f:
    caption_results = json.load(f)

captions = {r["item_id"]: r["caption"] for r in caption_results}
unified_items = build_unified_items_from_merged(merged_document, captions)

len(unified_items), unified_items[0]

## Chunk

In [ ]:
chunker = SemanticChunker()
chunks = chunker.chunk(unified_items, percentile=95.0, max_chars=6000)
chunker.unload()

len(chunks), [len(c) for c in chunks][:20]

## Inspect a sample chunk

In [ ]:
sample_chunk = chunks[len(chunks) // 2]
print(f"{len(sample_chunk)} items, item_ids: {[i.item_id for i in sample_chunk]}\n")
print(render_unified_markdown(sample_chunk))

## Save chunks

Each chunk saved as its rendered markdown plus the item_ids it spans —
the item_ids are what a later concept-extraction stage would attach to
extracted concepts for content linkage back to source.

In [ ]:
chunks_payload = [
    {
        "chunk_index": i,
        "item_ids": [item.item_id for item in chunk],
        "text": render_unified_markdown(chunk),
    }
    for i, chunk in enumerate(chunks)
]

chunks_path = OUTPUT_DIR / f"{PDF_STEM}_semantic_chunks.json"
with open(chunks_path, "w") as f:
    json.dump(chunks_payload, f, indent=2)

chunks_path